# 4장. PhysicsNeMo의 AI 모델 이해와 사용

## 개요

**학습 목표:** PhysicsNeMo와 GNN/Neural Operator/FNO/Transolver의 개념/특징/활용 이해

| 읽을 순서 | 알아볼 내용 |
|---|---|
| 1. PhysicsNeMo와 모델 사용 | 제공 모델의 역할, 그래프/격자/점 자료, 공통 온도 문제와 학습 과정 |
| 2. GNN | 노드/연결과 이웃 정보 전달 원리 |
| 3. Neural Operator와 FNO | 분포 사이의 관계를 배우는 원리와 정상상태 온도 예측 |
| 4. Transolver | 좌표가 있는 점들의 관계를 처리하는 구조와 활용 |
| 5. 다음 장 준비 | 모델 소개에서 후크 자료의 학습/검증으로 이어지는 흐름 |


## 1. PhysicsNeMo와 모델 사용 흐름


### 1.3 예제 실행 준비


### 04장 / 셀 03 / Notebook


In [ ]:
# 1. 현재 위치와 상위 폴더에서 교육 폴더를 찾습니다.
from pathlib import Path
import sys

working_dir = Path.cwd().resolve()
education_dir = next((p for base in [working_dir, *working_dir.parents]
                      for p in [base, base / 'education']
                      if (p / 'course_utils/dataset.py').is_file()), None)
if education_dir is None:
    raise FileNotFoundError('교육 폴더 또는 프로젝트 루트에서 실행하세요.')
if str(education_dir) not in sys.path:
    sys.path.insert(0, str(education_dir))

# 2. PhysicsNeMo를 불러오기 전에 교육용 임시 저장 위치를 준비합니다.
import os
os.environ["PHYSICSNEMO_FORCE_TE"] = "False"
os.environ.setdefault("WARP_CACHE_PATH", str(education_dir / "runs/.warp_cache"))
import course_utils
import warp
warp.config.log_level = warp.LOG_WARNING
import torch
import physicsnemo

# 3. 작은 예제의 실행 설정을 정하고 패키지 버전을 확인합니다.
torch.set_num_threads(4)
torch.manual_seed(42)
print('PyTorch:', torch.__version__, '| PhysicsNeMo:', physicsnemo.__version__)

import torch_geometric
print('PyG:', torch_geometric.__version__)
from physicsnemo.models.meshgraphnet import MeshGraphNet


### 1.4 세 모델로 풀어 볼 공통 온도 예측 문제


### 04장 / 셀 06 / Notebook


In [ ]:
# 1. 발열량과 열전도/경계조건으로 정상상태 온도 정답을 계산합니다.
import torch.nn.functional as F
import matplotlib.pyplot as plt

def solve_steady_temperature(source):
    # 가장자리를 제외한 30 × 30 위치의 열전도 균형식을 준비합니다.
    n = 30
    line = (2 * torch.eye(n, dtype=torch.float64)
            - torch.diag(torch.ones(n - 1, dtype=torch.float64), 1)
            - torch.diag(torch.ones(n - 1, dtype=torch.float64), -1))
    identity = torch.eye(n, dtype=torch.float64)
    matrix = (torch.kron(identity, line) + torch.kron(line, identity)) * 31 ** 2

    # 입력을 W/m³로 되돌려, 열전도율 1 W/(m·K)인 평판의 온도 상승을 구합니다.
    rhs = source[:, :, 1:-1, 1:-1].double().reshape(len(source), n * n) * 1000
    interior = torch.linalg.solve(matrix, rhs.T).T.reshape(len(source), 1, n, n)

    # 온도 상승을 40으로 나누고, 가장자리 20 ℃는 변환한 값 0으로 둡니다.
    target = torch.zeros_like(source)
    target[:, :, 1:-1, 1:-1] = (interior / 40).to(source.dtype)
    return target

# 2. 열이 발생하는 두 영역의 위치/크기/양을 바꾸어 자료를 만듭니다.
def make_source_temperature_pairs(count, seed):
    rng = torch.Generator().manual_seed(seed)
    y, x = torch.meshgrid(torch.linspace(0, 1, 32),
                          torch.linspace(0, 1, 32), indexing='ij')
    source = torch.zeros(count, 1, 32, 32)
    for _ in range(2):
        cx = 0.2 + 0.6 * torch.rand(count, 1, 1, 1, generator=rng)
        cy = 0.2 + 0.6 * torch.rand(count, 1, 1, 1, generator=rng)
        width = 0.06 + 0.12 * torch.rand(count, 1, 1, 1, generator=rng)
        amount = 0.2 + 0.3 * torch.rand(count, 1, 1, 1, generator=rng)
        source += amount * torch.exp(-((x - cx) ** 2 + (y - cy) ** 2) / (2 * width ** 2))
    source[:, :, [0, -1], :] = 0
    source[:, :, :, [0, -1]] = 0
    return source, solve_steady_temperature(source)

# 3. 모든 모델이 함께 사용할 CPU 원본을 한 번 준비합니다.
common_train_input, common_train_target = make_source_temperature_pairs(256, seed=4030)
common_val_input, common_val_target = make_source_temperature_pairs(64, seed=4031)
common_test_input, common_test_target = make_source_temperature_pairs(64, seed=4032)
common_pairs = [(common_train_input, common_train_target),
                (common_val_input, common_val_target),
                (common_test_input, common_test_target)]
print('공통 자료 | 학습:', tuple(common_train_input.shape),
      '| 검증:', tuple(common_val_input.shape), '| 시험:', tuple(common_test_input.shape))

# 4. 별도 CPU 난수 생성기로 에폭마다 학습 256개의 순서만 섞습니다.
# 세 모델은 같은 순서표를 16개씩 나누어 각 케이스를 에폭마다 정확히 한 번 사용합니다.
common_epochs, common_batch_size = 50, 16
common_order_generator = torch.Generator(device='cpu').manual_seed(4033)
common_epoch_orders = [torch.randperm(len(common_train_input), generator=common_order_generator)
                       for _ in range(common_epochs)]
common_steps_per_epoch = len(common_train_input) // common_batch_size
assert len(common_train_input) % common_batch_size == 0
assert all(torch.equal(order.sort().values, torch.arange(len(common_train_input)))
           for order in common_epoch_orders)
print(f'공통 학습 | 배치 {common_batch_size}개 | {common_epochs}에폭 | '
      f'에폭당 {common_steps_per_epoch}스텝 | 총 {common_epochs * common_steps_per_epoch}스텝')


## 2. GNN: 연결된 노드의 정보를 함께 처리하는 모델


### 2.5 PhysicsNeMo GNN으로 정상상태 온도 예측하기


#### 2.5.1 그래프와 모델 준비


### 04장 / 셀 12 / Notebook


In [ ]:
# 1. 공식 모델과 그래프 준비 도구를 불러옵니다.
from physicsnemo.models.meshgraphnet import MeshGraphNet
from course_utils.gnn_temperature import make_plate_graph, predict_temperature

# 2. 학습 배치 16개, 평가 배치 4개, 한 케이스용 그래프를 준비합니다.
gnn_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
graphs = {b: make_plate_graph(b, gnn_device) for b in (1, 4, common_batch_size)}
graph = graphs[1]
assert graph.num_nodes == 1024 and graph.num_edges == 9600
assert torch.equal(graph.batch[graph.edge_index[0]], graph.batch[graph.edge_index[1]])
print('한 케이스 노드:', graph.num_nodes, '| 방향을 구분한 엣지:', graph.num_edges)
print('노드 입력: (1024, 4) / 엣지 입력:', tuple(graph.edge_attr.shape))
print('계산 장치:', gnn_device)

# 3. 여섯 번의 정보 전달을 수행하는 작은 MeshGraphNet을 생성합니다.
torch.manual_seed(42)
gnn = MeshGraphNet(input_dim_nodes=4, input_dim_edges=3, output_dim=1,
    processor_size=6, hidden_dim_processor=32, hidden_dim_node_encoder=32,
    hidden_dim_edge_encoder=32, hidden_dim_node_decoder=32).to(gnn_device)

# 4. 같은 원본 자료를 사용하며, 평가는 메모리를 줄이기 위해 네 케이스씩 계산합니다.
gnn_train_x, gnn_train_y = [v.to(gnn_device) for v in common_pairs[0]]
gnn_val_x, gnn_val_y = [v.to(gnn_device) for v in common_pairs[1]]
gnn_test_x, gnn_test_y = [v.to(gnn_device) for v in common_pairs[2]]
def predict_gnn_cases(model, source):
    return torch.cat([predict_temperature(model, part, graphs[len(part)])
                      for part in source.split(4)])
gnn.eval()
with torch.no_grad():
    gnn_before = predict_gnn_cases(gnn, gnn_test_x).cpu()
print('학습/검증/시험 케이스:', len(gnn_train_x), len(gnn_val_x), len(gnn_test_x))
print('한 케이스 출력:', tuple(gnn_before[:1].shape))


#### 2.5.2 정답과 비교하며 학습


### 04장 / 셀 14 / Notebook


In [ ]:
# 1. 학습 256개를 에폭마다 한 번씩 사용해 50에폭 동안 가중치를 조정합니다.
gnn_optimizer = torch.optim.Adam(gnn.parameters(), lr=0.001)
gnn_loss_history, gnn_validation_history = [], []
for epoch, epoch_order in enumerate(common_epoch_orders, start=1):
    gnn.train()
    epoch_loss_sum = 0.0
    for start in range(0, len(epoch_order), common_batch_size):
        batch_ids = epoch_order[start:start + common_batch_size].to(gnn_device)
        gnn_optimizer.zero_grad()
        prediction = predict_temperature(gnn, gnn_train_x[batch_ids], graphs[len(batch_ids)])
        loss = F.mse_loss(prediction, gnn_train_y[batch_ids])
        loss.backward()
        gnn_optimizer.step()
        # 배치 평균에 케이스 수를 곱해 더한 뒤, 학습 256개 전체의 평균을 기록합니다.
        epoch_loss_sum += loss.detach().item() * len(batch_ids)
    gnn_loss_history.append(epoch_loss_sum / len(gnn_train_x))
    if epoch % 5 == 0:
        gnn.eval()
        with torch.no_grad():
            validation = predict_gnn_cases(gnn, gnn_val_x)
            val_mae = (validation - gnn_val_y).abs().mean().item() * 40
        gnn_validation_history.append(val_mae)
        print(f'{epoch}에폭 | 학습 평균 손실: {gnn_loss_history[-1]:.6f} | 검증 MAE: {val_mae:.3f} ℃')

# 2. 학습이 끝난 모델과 손실 곡선을 저장합니다.
gnn.eval()
demo_dir = education_dir / 'runs/model_intro'
demo_dir.mkdir(parents=True, exist_ok=True)
gnn_checkpoint = demo_dir / 'gnn_steady_temperature.mdlus'
gnn.save(gnn_checkpoint)
print('PhysicsNeMo 모델 저장:', gnn_checkpoint.name)
plt.rcParams.update({'font.family': 'Malgun Gothic', 'axes.unicode_minus': False})
fig, ax = plt.subplots(figsize=(8, 3), layout='constrained')
ax.plot(range(1, common_epochs + 1), gnn_loss_history)
ax.set(title='GNN의 정상상태 온도 예측 학습', xlabel='완료한 에폭(학습 자료 전체 사용 횟수)',
       ylabel='에폭 평균 학습 손실(MSE)', yscale='log')
# 로그 눈금을 일반 숫자로 표시해 지수의 마이너스 글꼴 경고를 방지합니다.
from matplotlib.ticker import FuncFormatter, NullFormatter
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, position: f'{value:g}'))
ax.yaxis.set_minor_formatter(NullFormatter())
ax.grid(alpha=0.3)
plt.show()


#### 2.5.3 시험 예측과 오차 확인


### 04장 / 셀 16 / Notebook


In [ ]:
# 1. 학습에 사용하지 않은 시험 64개에서 온도를 예측합니다.
with torch.no_grad():
    gnn_output = predict_gnn_cases(gnn, gnn_test_x).cpu()
gnn_source = common_test_input * 1000
gnn_target_c = common_test_target * 40 + 20
gnn_prediction_c = gnn_output * 40 + 20
gnn_before_c = gnn_before * 40 + 20
gnn_absolute_error = (gnn_prediction_c - gnn_target_c).abs()
gnn_before_mae = (gnn_before_c - gnn_target_c).abs().mean().item()
gnn_after_mae = gnn_absolute_error.mean().item()
gnn_baseline_mae = (20 - gnn_target_c).abs().mean().item()
assert gnn_output.shape == (64, 1, 32, 32) and torch.isfinite(gnn_output).all()
print(f'시험 64개 MAE(℃) | 모두 20 ℃: {gnn_baseline_mae:.3f} | 학습 전: {gnn_before_mae:.3f} | 학습 후: {gnn_after_mae:.3f}')

# 2. 같은 두 사례를 같은 위치에 배치합니다. 온도는 세 열 모두 같은 색 눈금입니다.
panels = [('발열량 입력(W/m³)', gnn_source, 'magma', 0, 1000),
          ('학습 전 예측(℃)', gnn_before_c, 'turbo', 20, 60),
          ('학습 후 예측(℃)', gnn_prediction_c, 'turbo', 20, 60),
          ('정상상태 정답(℃)', gnn_target_c, 'turbo', 20, 60)]
fig, axes = plt.subplots(2, 4, figsize=(13, 7), layout='constrained')
for col, (title, arrays, palette, low, high) in enumerate(panels):
    for row in range(2):
        image = axes[row, col].imshow(arrays[row, 0].numpy(), origin='lower',
            interpolation='nearest', cmap=palette, vmin=low, vmax=high)
        axes[row, col].set(title=title, aspect='equal')
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
        if col == 0: axes[row, col].set_ylabel(f'시험 사례 {row + 1}')
        fig.colorbar(image, ax=axes[row, col], shrink=0.75)
fig.suptitle('GNN: 같은 위치에서 비교한 발열량/온도 예측/정답', fontsize=16)
plt.show()


### 04장 / 셀 18 / Notebook


In [ ]:
# 3. 같은 두 사례에서 오차가 남은 위치를 확인합니다.
fig, axes = plt.subplots(1, 2, figsize=(9, 4), layout='constrained')
maximum = gnn_absolute_error[:2].max().item()
for i, ax in enumerate(axes):
    image = ax.imshow(gnn_absolute_error[i, 0].numpy(), origin='lower',
        interpolation='nearest', cmap='viridis', vmin=0, vmax=maximum)
    ax.set(title=f'시험 사례 {i + 1}: 온도 절대오차(℃)')
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(image, ax=ax, shrink=0.8)
plt.show()

# 4. 저장한 모델을 다시 불러와 첫 시험 사례의 예측이 같은지 확인합니다.
reloaded_gnn = MeshGraphNet.from_checkpoint(gnn_checkpoint).to(gnn_device).eval()
with torch.no_grad():
    restored = predict_temperature(reloaded_gnn, gnn_test_x[:1], graphs[1]).cpu()
torch.testing.assert_close(restored, gnn_output[:1], rtol=1e-4, atol=1e-5)
print('저장/재로딩 예측 일치: PASS')


#### 2.5.4 선택 확인: 한 노드의 입력과 연결 읽기


### 04장 / 셀 20 / Notebook


In [ ]:
# 선택 확인: 중앙 노드 528의 입력과 연결된 이웃을 읽습니다.
one_graph = make_plate_graph()
node_id = 16 * 32 + 16
neighbors = one_graph.edge_index[1, one_graph.edge_index[0] == node_id]
print('노드 번호(0부터):', node_id)
print('좌표:', one_graph.pos[node_id].tolist())
print('이웃 노드 번호:', neighbors.tolist())
print('발열량(W/m³):', gnn_source[0, 0, 16, 16].item())
print('예측/정답 온도(℃):', gnn_prediction_c[0, 0, 16, 16].item(),
      gnn_target_c[0, 0, 16, 16].item())


## 3. Neural Operator와 FNO


### 3.5 PhysicsNeMo FNO로 정상상태 온도 예측하기


#### 3.5.1 자료와 모델 준비


### 04장 / 셀 27 / Notebook


In [ ]:
# 1. GNN과 동일한 공통 원본에서 FNO용 발열량/온도 쌍을 준비합니다.
heat_train_input, heat_train_target = [a.clone() for a in common_pairs[0]]
heat_val_input, heat_val_target = [a.clone() for a in common_pairs[1]]
heat_test_input, heat_test_target = [a.clone() for a in common_pairs[2]]

# 2. PhysicsNeMo의 제공 FNO를 만들고 모델과 자료를 같은 장치에 올립니다.
from physicsnemo.models.fno import FNO
fno_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(44)
fno = FNO(in_channels=1, out_channels=1, dimension=2,
          latent_channels=16, num_fno_layers=2,
          num_fno_modes=8, padding=4).to(fno_device).eval()
heat_train_input, heat_train_target = heat_train_input.to(fno_device), heat_train_target.to(fno_device)
heat_val_input, heat_val_target = heat_val_input.to(fno_device), heat_val_target.to(fno_device)
fno_input = heat_test_input.to(fno_device)

# 3. 학습 전 예측과 실제 사용 모델/장치/자료의 모양을 확인합니다.
with torch.no_grad():
    fno_before = fno(fno_input).cpu()
device_name = torch.cuda.get_device_name(fno_device) if fno_device.type == 'cuda' else 'CPU'
print('사용 모델:', type(fno).__module__ + '.' + type(fno).__name__)
print('FNO 계산 장치:', fno_device, '|', device_name)
print('학습:', tuple(heat_train_input.shape), '| 검증:', tuple(heat_val_input.shape))
print('시험 입력/정답:', tuple(heat_test_input.shape))


#### 3.5.2 정답과 비교하며 학습


### 04장 / 셀 29 / Notebook


In [ ]:
# 1. 학습 256개를 에폭마다 한 번씩 사용해 50에폭 동안 가중치를 조정합니다.
heat_optimizer = torch.optim.Adam(fno.parameters(), lr=0.003)
heat_loss_history, heat_validation_history = [], []
for epoch, epoch_order in enumerate(common_epoch_orders, start=1):
    fno.train()
    epoch_loss_sum = 0.0
    for start in range(0, len(epoch_order), common_batch_size):
        batch_ids = epoch_order[start:start + common_batch_size].to(fno_device)
        heat_optimizer.zero_grad()
        prediction = fno(heat_train_input[batch_ids])
        loss = F.mse_loss(prediction, heat_train_target[batch_ids])
        loss.backward()
        heat_optimizer.step()
        # 배치 평균에 케이스 수를 곱해 더한 뒤, 학습 256개 전체의 평균을 기록합니다.
        epoch_loss_sum += loss.detach().item() * len(batch_ids)
    heat_loss_history.append(epoch_loss_sum / len(heat_train_input))
    if epoch % 5 == 0:
        fno.eval()
        with torch.no_grad():
            val_mae = float((fno(heat_val_input) - heat_val_target).abs().mean() * 40)
        heat_validation_history.append(val_mae)
        print(f'{epoch}에폭 | 학습 평균 손실: {heat_loss_history[-1]:.6f} | 검증 MAE: {val_mae:.3f} ℃')

# 5. 학습한 PhysicsNeMo 모델을 실습 전용 폴더에 저장합니다.
fno.eval()
heat_demo_dir = education_dir / 'runs/model_intro'
heat_demo_dir.mkdir(parents=True, exist_ok=True)
fno_checkpoint = heat_demo_dir / 'fno_steady_temperature.mdlus'
fno.save(fno_checkpoint)
print('PhysicsNeMo 모델 저장:', fno_checkpoint.name)

# 6. 학습에 따라 손실이 줄어드는지 그래프로 확인합니다.
plt.rcParams.update({'font.family': 'Malgun Gothic', 'axes.unicode_minus': False})
fig, ax = plt.subplots(figsize=(8, 3), layout='constrained')
ax.plot(range(1, common_epochs + 1), heat_loss_history)
ax.set(title='FNO의 정상상태 온도 예측 학습', xlabel='완료한 에폭(학습 자료 전체 사용 횟수)',
       ylabel='에폭 평균 학습 손실(MSE)')
ax.set_yscale('log')
ax.set_yticks([0.1, 0.01, 0.001, 0.0001, 0.00001],
             labels=['0.1', '0.01', '0.001', '0.0001', '0.00001'])
ax.grid(alpha=0.3)
plt.show()


#### 3.5.3 시험 예측과 오차 확인


### 04장 / 셀 32 / Notebook


In [ ]:
# 7. 시험 자료의 정상상태 온도를 예측하고 물리량별 단위로 되돌립니다.
with torch.no_grad():
    fno_output = fno(fno_input).cpu()
source_density = heat_test_input * 1000
steady_temperature = heat_test_target * 40 + 20
fno_before_temperature = fno_before * 40 + 20
fno_predicted_temperature = fno_output * 40 + 20
heat_absolute_error = (fno_predicted_temperature - steady_temperature).abs()
heat_baseline_mae = (20 - steady_temperature).abs().mean(dim=(1, 2, 3))
heat_before_mae = (fno_before_temperature - steady_temperature).abs().mean(dim=(1, 2, 3))
heat_after_mae = heat_absolute_error.mean(dim=(1, 2, 3))
print('시험 64개 평균 MAE(℃) | 모두 20 ℃인 기준값:', float(heat_baseline_mae.mean()),
      '| 학습 전:', float(heat_before_mae.mean()), '| 학습 후:', float(heat_after_mae.mean()))
for i in range(2):
    print(f'사례 {i + 1} | 학습 후 MAE: {heat_after_mae[i]:.3f} ℃')
assert fno_output.shape == (64, 1, 32, 32)
assert torch.isfinite(fno_output).all()

# 8. 발열량 입력과 온도 결과를 각 물리량의 눈금으로 비교합니다.
panels = [('발열량 입력(W/m³)', source_density, 'magma', 0, 1000),
          ('학습 전 예측(℃)', fno_before_temperature, 'turbo', 20, 60),
          ('학습 후 예측(℃)', fno_predicted_temperature, 'turbo', 20, 60),
          ('정상상태 정답(℃)', steady_temperature, 'turbo', 20, 60)]
fig, axes = plt.subplots(2, 4, figsize=(13, 7), layout='constrained')
for col, (title, arrays, palette, low, high) in enumerate(panels):
    for row in range(2):
        image = axes[row, col].imshow(arrays[row, 0].numpy(), origin='lower', cmap=palette,
                                     vmin=low, vmax=high, interpolation='nearest')
        axes[row, col].set_title(title)
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
        if col == 0:
            axes[row, col].set_ylabel(f'시험 사례 {row + 1}')
        fig.colorbar(image, ax=axes[row, col], shrink=0.75)
fig.suptitle('FNO의 정상상태 온도 예측: 학습에 쓰지 않은 시험 자료', fontsize=16)
plt.show()


### 04장 / 셀 34 / Notebook


In [ ]:
# 9. 예측과 정답의 차이가 어느 위치에 남았는지 ℃ 단위로 확인합니다.
fig, axes = plt.subplots(1, 2, figsize=(9, 4), layout='constrained')
error_max = float(heat_absolute_error[:2].max())
for i, ax in enumerate(axes):
    image = ax.imshow(heat_absolute_error[i, 0].numpy(), origin='lower', cmap='viridis',
                      vmin=0, vmax=error_max, interpolation='nearest')
    ax.set_title(f'시험 사례 {i + 1}: 온도 절대오차(℃)')
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(image, ax=ax, shrink=0.8)
fig.suptitle('예측/정답의 차이가 큰 위치 확인', fontsize=16)
plt.show()


#### 3.5.4 선택 확인: 중간 특징과 숫자 배열


### 04장 / 셀 37 / Notebook


In [ ]:
# 선택 1. 학습한 FNO에서 온도 출력 직전의 실제 중간 특징을 읽습니다.
with torch.no_grad():
    fno_features = fno.spec_encoder(fno_input[:2]).cpu()
assert fno_features.shape == (2, 16, 32, 32)
heat_feature_ids = fno_features.var(dim=(0, 2, 3)).topk(2).indices.tolist()
panels = [(f'중간 특징 {j + 1}', fno_features[:, j]) for j in heat_feature_ids]
fig, axes = plt.subplots(2, 2, figsize=(8, 7), layout='constrained')
for col, (title, arrays) in enumerate(panels):
    low, high = float(arrays.min()), float(arrays.max())
    for row in range(2):
        image = axes[row, col].imshow(arrays[row].numpy(), origin='lower', cmap='viridis',
                                     vmin=low, vmax=high, interpolation='nearest')
        axes[row, col].set_title(title)
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
        if col == 0:
            axes[row, col].set_ylabel(f'시험 사례 {row + 1}')
        fig.colorbar(image, ax=axes[row, col], shrink=0.75)
fig.suptitle('학습한 FNO의 출력 직전 중간 특징', fontsize=16)
plt.show()


### 04장 / 셀 39 / Notebook


In [ ]:
# 선택 2. 같은 위치의 발열량/중간 특징/온도 예측/정답을 비교합니다.
torch.set_printoptions(precision=3, sci_mode=False)
for i in range(2):
    print(f'시험 사례 {i + 1} | 가운데 3 × 3 영역')
    for name, array in [('발열량 입력(W/m³)', source_density[i, 0]),
                        (f'중간 특징 {heat_feature_ids[0] + 1}', fno_features[i, heat_feature_ids[0]]),
                        ('온도 예측(℃)', fno_predicted_temperature[i, 0]),
                        ('정상상태 온도 정답(℃)', steady_temperature[i, 0])]:
        print(name); print(array[15:18, 15:18])


## 4. Transolver: 점 정보를 모아 관계를 학습하는 모델


### 4.4 PhysicsNeMo Transolver로 점별 온도 예측하기


#### 4.4.1 자료와 모델 준비


### 04장 / 셀 45 / Notebook


In [ ]:
# 1. 기존 발열량/온도 쌍을 같은 순서의 점별 입력/정답으로 바꿉니다.
from physicsnemo.models.transolver import Transolver
transolver_pairs = [(q.clone(), t.clone()) for q, t in common_pairs]
point_pairs = [(q.flatten(2).transpose(1, 2).contiguous(),
                t.flatten(2).transpose(1, 2).contiguous()) for q, t in transolver_pairs]
y, x = torch.meshgrid(torch.linspace(0, 1, 32), torch.linspace(0, 1, 32), indexing='ij')
transolver_coords = torch.stack([x.flatten(), y.flatten()], dim=-1).unsqueeze(0)

# 2. 제공 모델과 모든 학습 자료를 같은 GPU(또는 CPU)에 올립니다.
transolver_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(45)
transolver = Transolver(functional_dim=1, embedding_dim=2, out_dim=1,
                        n_layers=3, n_hidden=64, n_head=4, slice_num=16,
                        structured_shape=None, use_te=False).to(transolver_device).eval()
transolver_coords = transolver_coords.to(transolver_device)
point_train_input, point_train_target = [a.to(transolver_device) for a in point_pairs[0]]
point_val_input, point_val_target = [a.to(transolver_device) for a in point_pairs[1]]
point_test_input, point_test_target = [a.to(transolver_device) for a in point_pairs[2]]

# 3. 여러 사례에서 같은 평판 좌표를 사용해 학습 전 예측을 보관합니다.
with torch.no_grad():
    point_before = transolver(point_test_input,
        embedding=transolver_coords.expand(len(point_test_input), -1, -1)).cpu()
device_name = torch.cuda.get_device_name(transolver_device) if transolver_device.type == 'cuda' else 'CPU'
print('사용 모델:', type(transolver).__module__ + '.' + type(transolver).__name__)
print('Transolver 계산 장치:', transolver_device, '|', device_name)
print('좌표:', tuple(transolver_coords.shape), '| 학습:', tuple(point_train_input.shape))
print('검증:', tuple(point_val_input.shape), '| 시험:', tuple(point_test_input.shape))
assert point_train_input.shape == point_train_target.shape == (256, 1024, 1)


#### 4.4.2 정답과 비교하며 학습


### 04장 / 셀 47 / Notebook


In [ ]:
# 1. 학습 256개를 에폭마다 한 번씩 사용해 50에폭 동안 가중치를 조정합니다.
point_optimizer = torch.optim.Adam(transolver.parameters(), lr=0.001)
point_loss_history, point_validation_history = [], []
for epoch, epoch_order in enumerate(common_epoch_orders, start=1):
    transolver.train()
    epoch_loss_sum = 0.0
    for start in range(0, len(epoch_order), common_batch_size):
        batch_ids = epoch_order[start:start + common_batch_size].to(transolver_device)
        point_optimizer.zero_grad()
        prediction = transolver(point_train_input[batch_ids],
                                    embedding=transolver_coords.expand(len(batch_ids), -1, -1))
        loss = F.mse_loss(prediction, point_train_target[batch_ids])
        loss.backward()
        point_optimizer.step()
        # 배치 평균에 케이스 수를 곱해 더한 뒤, 학습 256개 전체의 평균을 기록합니다.
        epoch_loss_sum += loss.detach().item() * len(batch_ids)
    point_loss_history.append(epoch_loss_sum / len(point_train_input))
    if epoch % 5 == 0:
        transolver.eval()
        with torch.no_grad():
            validation = transolver(point_val_input,
                embedding=transolver_coords.expand(len(point_val_input), -1, -1))
            val_mae = float((validation - point_val_target).abs().mean() * 40)
        point_validation_history.append(val_mae)
        print(f'{epoch}에폭 | 학습 평균 손실: {point_loss_history[-1]:.6f} | 검증 MAE: {val_mae:.3f} ℃')

# 5. 제공 모델을 저장하고 학습 손실의 변화를 확인합니다.
transolver.eval()
point_demo_dir = education_dir / 'runs/model_intro'
point_demo_dir.mkdir(parents=True, exist_ok=True)
transolver_checkpoint = point_demo_dir / 'transolver_steady_temperature.mdlus'
transolver.save(transolver_checkpoint)
print('PhysicsNeMo 모델 저장:', transolver_checkpoint.name)
plt.rcParams.update({'font.family': 'Malgun Gothic', 'axes.unicode_minus': False})
fig, ax = plt.subplots(figsize=(8, 3), layout='constrained')
ax.plot(range(1, common_epochs + 1), point_loss_history)
ax.set(title='Transolver의 점별 온도 예측 학습', xlabel='완료한 에폭(학습 자료 전체 사용 횟수)',
       ylabel='에폭 평균 학습 손실(MSE)', yscale='log')
ax.set_yticks([0.1, 0.01, 0.001, 0.0001], labels=['0.1', '0.01', '0.001', '0.0001'])
ax.grid(alpha=0.3)
plt.show()


#### 4.4.3 시험 예측과 오차 확인


### 04장 / 셀 50 / Notebook


In [ ]:
# 6. 학습에 사용하지 않은 점별 시험 입력에서 온도를 예측합니다.
with torch.no_grad():
    point_output = transolver(point_test_input,
        embedding=transolver_coords.expand(len(point_test_input), -1, -1)).cpu()
point_source = point_test_input.cpu() * 1000
point_temperature = point_test_target.cpu() * 40 + 20
point_prediction = point_output * 40 + 20
point_before_temperature = point_before * 40 + 20
point_absolute_error = (point_prediction - point_temperature).abs()
point_before_mae = (point_before_temperature - point_temperature).abs().mean(dim=(1, 2))
point_after_mae = point_absolute_error.mean(dim=(1, 2))
point_baseline_mae = (20 - point_temperature).abs().mean(dim=(1, 2))
print('시험 64개 평균 MAE(℃) | 모두 20 ℃인 기준값:', float(point_baseline_mae.mean()),
      '| 학습 전:', float(point_before_mae.mean()), '| 학습 후:', float(point_after_mae.mean()))
for i in range(2):
    print(f'사례 {i + 1} | 학습 후 MAE: {point_after_mae[i]:.3f} ℃')
assert point_output.shape == (64, 1024, 1)
assert torch.isfinite(point_output).all()

# 7. 점별 값을 원래의 32 × 32 위치에 배치해 빈틈 없는 색상 격자로 비교합니다.
# 모델은 점 목록을 처리하며, 아래 reshape는 그림을 그리기 위한 배열 정리입니다.
panels = [('발열량 입력(W/m³)', point_source, 'magma', 0, 1000),
          ('학습 전 예측(℃)', point_before_temperature, 'turbo', 20, 60),
          ('학습 후 예측(℃)', point_prediction, 'turbo', 20, 60),
          ('정상상태 정답(℃)', point_temperature, 'turbo', 20, 60)]
fig, axes = plt.subplots(2, 4, figsize=(13, 7), layout='constrained')
for col, (title, arrays, palette, low, high) in enumerate(panels):
    for row in range(2):
        image = axes[row, col].imshow(arrays[row, :, 0].reshape(32, 32).numpy(),
            origin='lower', interpolation='nearest', cmap=palette, vmin=low, vmax=high)
        axes[row, col].set(title=title, aspect='equal')
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
        if col == 0:
            axes[row, col].set_ylabel(f'시험 사례 {row + 1}')
        fig.colorbar(image, ax=axes[row, col], shrink=0.75)
fig.suptitle('Transolver: 같은 점에서 비교한 입력/예측/정답', fontsize=16)
plt.show()


### 04장 / 셀 52 / Notebook


In [ ]:
# 8. 점별 오차도 원래의 32 × 32 위치에 배치해 같은 색 눈금으로 비교합니다.
fig, axes = plt.subplots(1, 2, figsize=(9, 4), layout='constrained')
error_max = float(point_absolute_error[:2].max())
for i, ax in enumerate(axes):
    image = ax.imshow(point_absolute_error[i, :, 0].reshape(32, 32).numpy(),
        origin='lower', interpolation='nearest', cmap='viridis', vmin=0, vmax=error_max)
    ax.set(title=f'시험 사례 {i + 1}: 점별 온도 절대오차(℃)', aspect='equal')
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(image, ax=ax, shrink=0.8)
fig.suptitle('점별 예측 오차가 남은 위치 확인', fontsize=16)
plt.show()
